In [15]:
import random
from collections import deque

In [16]:
class Tile:
    
    def __init__(self, tuple_tile: tuple[int, int]):
        self.left_side = tuple_tile[0]
        self.right_side = tuple_tile[1]


    def __str__(self) -> str:
        return f"[{self.left_side}|{self.right_side}]"


    def __repr__(self) -> str:
        return self.__str__()


    def __eq__(self, other: Tile) -> bool:
        if not isinstance(other, Tile):
            return NotImplemented
        return (
            (self.left_side == other.left_side 
             and self.right_side == other.right_side)
            or 
            (self.left_side == other.right_side 
             and self.right_side == other.left_side)
        )


    def __iter__(self):
        yield self.left_side
        yield self.right_side


    def __add__(self, other):
        if isinstance(other, Tile):
            return (
                self.left_side + self.right_side
                + other.left_side + other.right_side
            )
        return NotImplemented


    @staticmethod
    def swap_tile(tile: Tile) -> Tile:
        tile.left_side, tile.right_side = tile.right_side, tile.left_side
        return tile

In [17]:
class Table:

    def __init__(self):
        self.tiles_deque = deque()
        self.table_left_side = None
        self.table_right_side = None


    def __str__(self) -> str:
        if not self.tiles_deque:
            return "Mesa vazia"

        return "".join(str(tile) for tile in self.tiles_deque)


    def __contains__(self, tile: Tile):
        return tile in self.tiles_deque


    def __getitem__(self, index: int) -> Tile:
        return self.tiles_deque[index]


    def add_first_tile(self, initial_tile: Tile):
        self.table_left_side = initial_tile.left_side
        self.table_right_side = initial_tile.right_side

        self.tiles_deque.append(initial_tile)


    def add_left(self, tile: Tile) -> None:
        if tile.right_side != self.table_left_side:
            if tile.left_side != self.table_left_side:
                raise ValueError("Não vale roubar!!")

            Tile.swap_tile(tile)

        self.tiles_deque.appendleft(tile)
        self.table_left_side = tile.left_side


    def add_right(self, tile: Tile) -> None:
        if tile.left_side != self.table_right_side:
            if tile.right_side != self.table_right_side:
                raise ValueError("Não vale roubar!!")

            Tile.swap_tile(tile)

        self.tiles_deque.append(tile)
        self.table_right_side = tile.right_side


    def show_table(self):
        print(self)


In [18]:
class Player:

    def __init__(self, name: str, tiles: list[Tile]):
        self.name = name
        self.tiles = tiles
        self.wins = 0


    def __str__(self):
        return f"{self.name}\n{self.tiles}"


    def __repr__(self) -> str:
        return f"{self.name}\n{self.tiles}"


    def get_tile_points(self) -> int:
        return sum(tile.left_side + tile.right_side for tile in self.tiles)


    def can_play(self, table: Table) -> bool:

        if not self.tiles:
            return False

        if not table.tiles_deque:
            return True

        for t in self.tiles:
            if (
                t.left_side == table.table_left_side or t.left_side == table.table_right_side 
                    or 
                t.right_side == table.table_right_side or t.right_side == table.table_left_side
            ):
                return True
            
        return False


    def get_playable_tiles(self, table: Table, verbose: bool = False):

        if not table.tiles_deque:
            playable_tiles = self.tiles.copy()
        else:
            playable_tiles = [
                t for t in self.tiles 
                if (
                    t.left_side in (
                        table.table_left_side,
                        table.table_right_side
                    )   
                    or t.right_side in(
                        table.table_left_side,
                        table.table_right_side
                    )
                )
            ]

        if verbose and playable_tiles:
            width = 8

            print("".join(str(tile).center(width) for tile in playable_tiles))
            print("".join(str(idx).center(width) for idx in range(len(playable_tiles))))

        return playable_tiles
    

    def choice_tile(self, table: Table, playable_tiles: list[Tile]):

        LEFT = {"l", "left", "e", "esquerda"}
        RIGHT = {"r", "right", "d", "direita"}

        tile_idx = int(input("Digite o número equivalente da peça: "))

        if not 0 <= tile_idx < len(playable_tiles):
            raise IndexError("Escolha um índice válido")

        side = input("Digite r (direita) ou l (esquerda) para dedicir onde botar a peça: ").strip().lower()

        if side not in LEFT | RIGHT:
            raise ValueError("Escolha um lado correto")

        tile = playable_tiles[tile_idx]

        if not table.tiles_deque:
            table.add_first_tile(tile)
        elif side in LEFT:
            table.add_left(tile)
        elif side in RIGHT:
            table.add_right(tile)


    def play_turn(self, table: Table):

        table.show_table()

        # passar turno
        if not self.can_play(table):
            return

        playable_tiles = self.get_playable_tiles(table, verbose=True)

        while True:
            try:
                self.choice_tile(table, playable_tiles)
                break
            except (ValueError, IndexError):
                print("Escolha outra peça")
 

In [19]:
YES = {"y", "yes", "yah", "s", "sim"}
NOT = {"n", "no", "not", "nao"}

In [20]:
class Domino:

    def __init__(self):
        self.players = []
        self.sleep = None

        self.table = Table()

        self.tiles = Domino.make_tiles()
        self.TOTAL_TILES = len(self.tiles)
            


    @staticmethod
    def make_tiles():
        tiles_tup = []
        pivot = 0
        for side1 in range(0, 7):
            for side2 in range(pivot, 7):
                tiles_tup.append((side1, side2))
            pivot+=1
        tiles = [Tile(tup) for tup in tiles_tup]
        return tiles


    @staticmethod
    def get_generic_doubles(tiles_list: list[Tile] | None = None) -> list[Tile]:
        if tiles_list is None:
            tiles_list = Domino.make_tiles()

        return [t for t in tiles_list if t.left_side == t.right_side]


    @staticmethod
    def get_generic_not_doubles(tiles_list: list[Tile] | None = None) -> list[Tile]:
        if tiles_list is None:
            tiles_list = Domino.make_tiles()

        return [t for t in tiles_list if t.left_side != t.right_side]


    @staticmethod
    def get_generic_prioritized_tiles(tiles_list: list[Tile] | None = None) -> list[Tile]:
        if tiles_list is None:
            tiles_list = Domino.make_tiles()

        return (
            Domino.get_generic_doubles(tiles_list)[::-1] +
            sorted(
                Domino.get_generic_not_doubles(tiles_list), 
                key=lambda tile: sum(tile), 
                reverse=True
            )    
        )


    def divide_tiles(
            self,
            n_players = 4, 
            tile_count = 7,
            has_sleep = True
        ) -> list[Player]:
        
        if 4 < n_players or n_players < 2:
            raise ValueError("Só é permitido entre 2 a 4 jogadores")
        if tile_count < 5 or len(self.tiles) // n_players < tile_count:
            raise ValueError("Selecione uma quantidade válida de peças")
        
        random.shuffle(self.tiles)

        self.players = [
            Player(
                f"P{n+1}", 
                [self.tiles.pop() for _ in range(tile_count)]
            ) for n in range(n_players)]

        if not has_sleep or len(self.tiles) == 1:
            self.sleep = None
        else:
            self.sleep = self.tiles.copy()
            self.tiles.clear()

        return self.players


    def find_first_player_with_tile(self, tile_list: list[Tile]) -> tuple[Player, Tile]:
        for tile in tile_list:
            for player in self.players:
                if tile in player.tiles:
                    return player, tile
        raise ValueError("Nenhum jogador tem as peças")


    def get_match_podium(self, verbose: bool = False) -> list[Player]:
        podium = sorted(self.players, key=lambda player: player.get_tile_points())

        if verbose:
            print("-"*50)
            pos = 1
            for p in podium:
                print(f"{pos}º - {p.name} - {p.get_tile_points()} points")
                pos += 1
            print("-"*50)

        return podium


    def get_game_podium(self, verbose: bool = False) -> list[Player]:
        podium = sorted(self.players, key=lambda player: player.wins)
        
        if verbose:
            print("~=~"*30)
            pos = 1
            for p in podium:
                print(f"{pos}º - {p.name} - {p.wins} wins")
                pos += 1
            print("~=~"*30)

        return podium


    def find_player_with_fewer_points(self) -> Player | None:

        players_ordened_per_points = self.get_match_podium()

        if players_ordened_per_points[0].get_tile_points() == players_ordened_per_points[1].get_tile_points():
            return None

        return players_ordened_per_points[0]
    

    def rotate_players(self, starter: Player) -> None:
        starter_idx = self.players.index(starter)
        self.players = self.players[starter_idx:] + self.players[:starter_idx]


    def verify_starter(self) -> tuple[Player, Tile]:

        prioritized_tiles = self.get_generic_prioritized_tiles()

        try:
            starter, starter_tile = self.find_first_player_with_tile(prioritized_tiles)

        except ValueError:
            starter = random.choice(self.players)
            starter_tile = max(
                starter.tiles, 
                key=lambda tile: tile.left_side + tile.right_side
            )

        self.rotate_players(starter)
        return starter, starter_tile


    def verify_win(self) -> bool:
        for player in self.players:
            if not player.tiles:
                return True
        return False


    def game_is_locked(self) -> bool:
        for player in self.players:
            if player.can_play(self.table):
                return False
        return True


    def has_winner_or_draw(self, player: Player):
        if self.verify_win():
            player.wins+=1
            print(f"{player.name} venceu!!!")
            return True
        
        if self.game_is_locked():
            winner = self.find_player_with_fewer_points()

            if winner == None:
                print("Empate por pontos.")
                return True
            else:

                winner.wins+=1

                print(f"{winner} venceu por pontos!!!")

                self.get_match_podium(verbose=True)
                return True
        return False


    def ask_new_round(self):

        repeat = input("Deseja jogar novamente? ").strip().lower()
        
        if repeat in NOT:
            return False
        if repeat in YES:
            return True


    def define_match_configuration(self):
        
        n_players = int(input("Digite a quantidade de jogadores (2 a 4): "))
        tile_count = int(input("Digite a quantidade de peças para cada jogador (5 a 28/players): "))
        has_sleep_str = input("Terá dorme|puxa|compra? (y|n): ").strip().lower()

        if has_sleep_str in YES:
            has_sleep = True
        elif has_sleep_str in NOT:
            has_sleep = False
        else:
            raise ValueError("Digite y ou n")

        return n_players, tile_count, has_sleep
        

    def reset_game(self):
        self.players = []
        self.sleep = None
        self.table = Table()
        self.tiles = Domino.make_tiles()


    def start_game(self):

        while True:
            try:
                n_players, tile_count, has_sleep = self.define_match_configuration()
                break
            except ValueError as error:
                print(error)

        while True:

            self.divide_tiles(
                n_players,
                tile_count,
                has_sleep
            )

            self.verify_starter()

            game_ended = False
            while not game_ended:
                for player in self.players:
                    player.play_turn(self.table)

                    if self.has_winner_or_draw(player):
                        self.get_game_podium(verbose=True)
                        game_ended = True
                        break

            if not self.ask_new_round():
                return
            
            self.reset_game()


In [ ]:
#TODO 
# - falta espaçar corretamente um print de outro;
# - tem q printar além das peças jogaveis, todas as peças do jogador;
# - resolver o bug que a peça não é retirada da mão do jogador;
# - adicionar o dorme funcional e printar caso ele esteja no jogo
# - deve (com certeza) ter mais bugs
# - treinar a IA

In [ ]:
domino = Domino()

domino.start_game()



Mesa vazia
 [2|6]   [1|3]   [3|5]   [6|6]   [1|1]   [5|5]   [0|4]   [4|4]   [1|6]   [2|2]  
   0       1       2       3       4       5       6       7       8       9    
[6|6]
 [3|6]  
   0    
[6|6][6|3]
 [2|6]   [1|3]   [3|5]   [6|6]   [1|6]  
   0       1       2       3       4    
Escolha outra peça


KeyboardInterrupt: Interrupted by user